# TUMSOEV Blender + WanGP Studio — 5 секунд, 9:16

Этот ноутбук использует **настоящий Blender** в Google Colab: извлекает движение из референс-видео, создаёт 3D-блокинг, рендерит pose-control и затем запускает WanGP. Платные API и кредиты не используются.

Запускайте клетки сверху вниз. Первый тест ограничен 5 секундами.

In [ ]:
# Безопасные настройки первого теста
USE_DRIVE = False          # True сохраняет веса и результаты в Google Drive
REFERENCE_DURATION = 5.0   # не увеличивать без отдельного решения
CONTROL_FPS = 24
OUTPUT_WIDTH = 432
OUTPUT_HEIGHT = 768
print('Профиль: бесплатный тест, 5 секунд, 9:16, без платных API.')

## 1. GPU и WanGP

В метаданных ноутбука уже указан GPU. Если Colab всё равно показывает CPU: **Runtime → Change runtime type → T4 GPU → Save**.

In [ ]:
import json, subprocess, sys
from pathlib import Path

subprocess.run(['nvidia-smi'], check=True)
UPSTREAM_DIR = Path('/content/Wan2GP-on-Colab')
UPSTREAM_COMMIT = 'e428b5ebc0d49589474ef5d81e05cc2ab3c1e17b'
if not (UPSTREAM_DIR / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/Square-Zero-Labs/Wan2GP-on-Colab.git', str(UPSTREAM_DIR)], check=True)
subprocess.run(['git', '-C', str(UPSTREAM_DIR), 'fetch', '--depth', '1', 'origin', UPSTREAM_COMMIT], check=True)
subprocess.run(['git', '-C', str(UPSTREAM_DIR), 'checkout', '--detach', UPSTREAM_COMMIT], check=True)
upstream_notebook = UPSTREAM_DIR / 'wan2gp-google-colab.ipynb'
upstream = json.loads(upstream_notebook.read_text(encoding='utf-8'))
launch_code = None
for number, cell in enumerate(upstream['cells']):
    if cell.get('cell_type') != 'code':
        continue
    code = ''.join(cell.get('source') or [])
    if 'Launching Wan2GP' in code:
        launch_code = code
        continue
    code = code.replace('USE_GOOGLE_DRIVE_DATA = False', f'USE_GOOGLE_DRIVE_DATA = {USE_DRIVE!r}')
    print(f'\n--- WanGP setup cell {number} ---')
    exec(compile(code, f'{upstream_notebook.name}:cell_{number}', 'exec'), globals())
if launch_code is None:
    raise RuntimeError('WanGP launch cell was not found in the verified upstream notebook.')
print('WanGP подготовлен. Финальный запуск будет после Blender-блокинга.')

## 2. Установка Blender и трекера движения

Blender работает без окна, но создаёт настоящий `.blend`, анимацию объектов, камеру и MP4 control-video.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

env = os.environ.copy()
env['DEBIAN_FRONTEND'] = 'noninteractive'
subprocess.run(['sudo', 'apt-get', 'update', '-qq'], check=True, env=env)
subprocess.run(['sudo', 'apt-get', 'install', '-y', '--no-install-recommends', 'blender', 'python3-venv'], check=True, env=env)

STUDIO_REPO = Path('/content/tumsoev-blender-studio')
STUDIO_BRANCH = 'main'
if (STUDIO_REPO / '.git').exists():
    subprocess.run(['git', '-C', str(STUDIO_REPO), 'fetch', 'origin', STUDIO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(STUDIO_REPO), 'checkout', STUDIO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(STUDIO_REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', STUDIO_BRANCH, 'https://github.com/magomedt149/nova-robot.git', str(STUDIO_REPO)], check=True)

POSE_ENV = Path('/content/tumsoev-pose-env')
POSE_PYTHON = POSE_ENV / 'bin/python'
if not POSE_PYTHON.exists():
    subprocess.run([sys.executable, '-m', 'venv', str(POSE_ENV)], check=True)
probe = subprocess.run([str(POSE_PYTHON), '-c', 'import cv2, mediapipe'], capture_output=True)
if probe.returncode != 0:
    subprocess.run([str(POSE_PYTHON), '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
    subprocess.run([str(POSE_PYTHON), '-m', 'pip', 'install', '-q', 'mediapipe', 'opencv-python-headless'], check=True)
subprocess.run(['blender', '--version'], check=True)
print('Blender и pose tracker готовы.')

## 3. Загрузите референс-видео

Выберите один ролик. Лучше: полный рост, человек не закрыт предметами, камера не режет руки и ноги. Будут обработаны только первые 5 секунд.

In [ ]:
import os, shutil, subprocess
from pathlib import Path
from google.colab import files
from IPython.display import FileLink, Video, display

uploaded = files.upload()
video_names = [name for name in uploaded if Path(name).suffix.lower() in {'.mp4', '.mov', '.mkv', '.webm', '.avi'}]
if not video_names:
    raise RuntimeError('Видео не выбрано. Повторите клетку и загрузите MP4/MOV/MKV/WEBM/AVI.')
source_name = video_names[0]
source_video = Path('/content') / Path(source_name).name
source_video.write_bytes(uploaded[source_name])

CONTROL_DIR = Path('/content/TUMSOEV_CONTROL')
if CONTROL_DIR.exists():
    shutil.rmtree(CONTROL_DIR)
CONTROL_DIR.mkdir(parents=True)
reference_control = CONTROL_DIR / 'reference_control_9x16.mp4'
motion_json = CONTROL_DIR / 'motion.json'
ffmpeg = os.environ.get('FFMPEG_BINARY') or shutil.which('ffmpeg')
filter_graph = (
    f'scale={OUTPUT_WIDTH}:{OUTPUT_HEIGHT}:force_original_aspect_ratio=decrease,'
    f'pad={OUTPUT_WIDTH}:{OUTPUT_HEIGHT}:(ow-iw)/2:(oh-ih)/2:black,fps={CONTROL_FPS}'
)
subprocess.run([ffmpeg, '-y', '-i', str(source_video), '-t', str(REFERENCE_DURATION), '-vf', filter_graph, '-an', '-c:v', 'libx264', '-pix_fmt', 'yuv420p', str(reference_control)], check=True)
scripts = STUDIO_REPO / 'blender-colab/scripts'
subprocess.run([str(POSE_PYTHON), str(scripts / 'extract_pose.py'), '--input', str(reference_control), '--output', str(motion_json), '--fps', str(CONTROL_FPS), '--duration', str(REFERENCE_DURATION)], check=True)
subprocess.run(['blender', '--background', '--python', str(scripts / 'make_blender_control.py'), '--', '--motion', str(motion_json), '--output-dir', str(CONTROL_DIR), '--width', str(OUTPUT_WIDTH), '--height', str(OUTPUT_HEIGHT)], check=True)
pose_control = CONTROL_DIR / 'blender_pose_control.mp4'
if not pose_control.exists():
    candidates = sorted(CONTROL_DIR.glob('blender_pose_control*.mp4'))
    if candidates:
        candidates[0].replace(pose_control)
if not pose_control.exists():
    raise RuntimeError('Blender finished but the pose-control video was not found.')
archive_base = Path('/content/TUMSOEV_Blender_Control_5s')
archive_path = Path(shutil.make_archive(str(archive_base), 'zip', root_dir=CONTROL_DIR))
print('Готово:', CONTROL_DIR)
display(Video(str(reference_control), embed=True, width=320))
display(Video(str(pose_control), embed=True, width=320))
display(FileLink(str(archive_path)))

## 4. Запуск WanGP

После появления ссылки Gradio откройте её. Для первого теста используйте **Wan 2.2 Animate 2**: фото персонажа как reference image и `reference_control_9x16.mp4` как driving/reference video. Если нужен именно скелетный контроль, используйте `blender_pose_control.mp4`.

Не увеличивайте длительность и разрешение первого запуска: бесплатной T4 обычно хватает только на облегчённый 5-секундный профиль.

In [ ]:
# Эта клетка остаётся запущенной, пока открыт WanGP.
exec(compile(launch_code, 'verified_wan2gp_launch_cell', 'exec'), globals())